# 01 · Soma playground

The somatic compartment **on its own**: no dendrite yet. This is step 1 of three:

1. **Soma alone** (this notebook): leak, threshold, reset and adaptation
2. **Dendrite alone** (notebook 02): the sigmoid plateau, triggered by a hand-made bAP pulse
3. **Coupled** (notebook 03): the bAP comes from real somatic spikes and the plateau drives the soma

The equation (docs/model.md §1):

$$\frac{du_s}{dt} = -\frac{u_s - E_L}{\tau_s} + \frac{I_{\text{inj}} + I_{\text{noise}} + w_s}{C_s}, \qquad \frac{dw_s}{dt} = -\frac{w_s}{\tau_{w_s}}$$

A spike happens when $u_s > -50$ mV. Then $u_s \to E_L$, $w_s \to w_s + b_{w_s}$, and a 3 ms refractory period follows.

The dendritic drive $g_s f(u_d)$ is just a current into the soma, so in experiment 4 we fake it with a rectangular pulse.

**Reading:** Neuronal Dynamics 1.3, 6.1 and 6.2.

**Running on Colab:** run the first code cell. It installs Brian2 and loads `src/`, either from files you upload (`soma.py`, `plot_style.py`) or by cloning the repo.

In [ ]:
import os, sys

# Works both locally (repo checkout) and on Google Colab.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q brian2
    REPO_URL = "https://github.com/Muhanned7/naud-burst-multiplexing.git"
    if os.path.exists("/content/soma.py"):           # option A: you uploaded src/*.py to the Files panel
        sys.path.append("/content")
    else:                                            # option B: clone the repo
        if not os.path.exists("/content/repo"):
            !git clone -q {REPO_URL} /content/repo
        sys.path.append("/content/repo/src")
else:
    sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from brian2 import ms, mV, pA

from soma import SOMA_PARAMS, simulate_soma, step, plateau, isis, rheobase_pA, with_spikes_drawn
from plot_style import apply_style, COLORS, INK_SOFT

apply_style()
V_T = float(SOMA_PARAMS["V_T"] / mV)

def plot_run(r, title):
    # Three stacked panels on one time axis: input current, voltage, adaptation current.
    fig, (ax_i, ax_u, ax_w) = plt.subplots(
        3, 1, figsize=(9, 5.2), sharex=True, gridspec_kw={"height_ratios": [1, 2.4, 1]})
    ax_i.plot(r["t"], r["I"], color=COLORS["input"])
    ax_i.set_ylabel("input (pA)")
    ax_i.set_title(title)

    ax_u.plot(r["t"], with_spikes_drawn(r), color=COLORS["soma"], lw=1.1)
    ax_u.axhline(V_T, color=INK_SOFT, lw=0.8, ls="--")
    ax_u.text(r["t"][-1], V_T + 1, "threshold", ha="right", va="bottom", fontsize=8, color=INK_SOFT)
    ax_u.set_ylabel("soma u_s (mV)")

    ax_w.plot(r["t"], r["w"], color=COLORS["adapt"])
    ax_w.set_ylabel("w_s (pA)")
    ax_w.set_xlabel("time (ms)")
    fig.align_ylabels()
    plt.tight_layout()
    plt.show()
    print(f"{len(r['spikes'])} spikes · ISIs (ms): {np.round(isis(r['spikes']), 1)}")

print(f"Rheobase (minimum current that ever fires): {rheobase_pA():.0f} pA")

## 1 · One step of current

The rheobase is $(V_T - E_L)\,C_s/\tau_s = 20\text{ mV} \times 370\text{ pF} / 16\text{ ms} \approx 463$ pA. Anything below it never fires, however long you wait.

Watch three things below:
- the voltage **charges up** toward a steady level (the leak),
- each spike **resets** it to −70 mV,
- $w_s$ **steps down** by 200 pA per spike and decays over about 100 ms. That's the hyperpolarising brake.

In [ ]:
r = simulate_soma(step(600, 50 * ms, 550 * ms, 600 * ms))
plot_run(r, "600 pA step: the soma fires, then slows down")

## 2 · Stronger input means faster firing

The same step at four amplitudes. 400 pA is below rheobase and stays silent. Notice that the spacing between spikes *grows* in every trace. That's adaptation.

In [ ]:
amps = [400, 500, 700, 1000]
fig, axes = plt.subplots(len(amps), 1, figsize=(9, 6), sharex=True, sharey=True)
for ax, amp in zip(axes, amps):
    r = simulate_soma(step(amp, 50 * ms, 550 * ms, 600 * ms))
    ax.plot(r["t"], with_spikes_drawn(r), color=COLORS["soma"], lw=1.0)
    ax.axhline(V_T, color=INK_SOFT, lw=0.8, ls="--")
    ax.set_title(f"{amp} pA · {len(r['spikes'])} spikes", fontsize=9)
    ax.set_ylabel("mV")
axes[-1].set_xlabel("time (ms)")
plt.tight_layout()
plt.show()

## 3 · Adaptation on vs off

Set $b_{w_s} = 0$ and the brake disappears: intervals stay constant. On the left, each interval is plotted against its position in the spike train. On the right is the f–I curve (steady firing rate against input current).

In [ ]:
no_adapt = {"b_w_s": 0 * pA}
r_on = simulate_soma(step(1000, 0 * ms, 500 * ms, 500 * ms))
r_off = simulate_soma(step(1000, 0 * ms, 500 * ms, 500 * ms), params=no_adapt)

currents = np.arange(0, 1601, 100)
def steady_rate(I, params=None):
    # Rate over the last 500 ms of a 1 s step, after adaptation has settled.
    sp = simulate_soma(step(I, 0 * ms, 1000 * ms, 1000 * ms), params=params)["spikes"]
    return np.sum(sp >= 500) / 0.5
rate_on = [steady_rate(I) for I in currents]
rate_off = [steady_rate(I, no_adapt) for I in currents]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
for r, c, name in [(r_on, COLORS["soma"], "adaptation on"), (r_off, COLORS["alt"], "adaptation off")]:
    d = isis(r["spikes"])
    ax1.plot(np.arange(1, len(d) + 1), d, "o-", color=c, ms=5, label=name)
    ax1.annotate(name, (len(d), d[-1]), xytext=(4, 0), textcoords="offset points", va="center", fontsize=8, color=INK_SOFT)
ax1.set_xlabel("interval number")
ax1.set_ylabel("ISI (ms)")
ax1.set_title("1000 pA step: intervals")
ax1.legend(loc="center right")

ax2.plot(currents, rate_on, "o-", color=COLORS["soma"], ms=5, label="adaptation on")
ax2.plot(currents, rate_off, "o-", color=COLORS["alt"], ms=5, label="adaptation off")
ax2.axvline(rheobase_pA(), color=INK_SOFT, lw=0.8, ls="--")
ax2.text(rheobase_pA() + 20, max(rate_off) * 0.45, "rheobase", fontsize=8, color=INK_SOFT)
for rates, name in [(rate_on, "adaptation on"), (rate_off, "adaptation off")]:
    ax2.annotate(name, (currents[-1], rates[-1]), xytext=(-4, 8), textcoords="offset points", ha="right", fontsize=8, color=INK_SOFT)
ax2.set_xlabel("input current (pA)")
ax2.set_ylabel("steady rate (Hz)")
ax2.set_title("f–I curve")
ax2.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 4 · Preview: a fake dendritic plateau makes a burst

When the dendrite fires its plateau, the soma receives up to $g_s = 1300$ pA for about 10–40 ms. Here we inject that by hand on top of a weak 500 pA background:

- **Before the pulse:** the soma fires a single spike now and then (an isolated spike).
- **During the pulse:** spikes come about 9 ms apart, under the 16 ms burst criterion. **That is a burst.**

In the real model the pulse isn't hand-made. The dendrite produces it, and only when top-down input has primed it (notebook 02).

In [ ]:
duration = 300 * ms
I = step(500, 0 * ms, 300 * ms, duration) + plateau(1300, 150 * ms, 25 * ms, duration)
r = simulate_soma(I)
plot_run(r, "500 pA background + 1300 pA, 25 ms fake plateau at 150 ms")
burst_isis = isis(r["spikes"])
print("ISIs under 16 ms (part of a burst):", np.round(burst_isis[burst_isis < 16], 1))

## 5 · Background noise

The paper adds Ornstein–Uhlenbeck noise ($\tau_{\text{OU}} = 2$ ms) sized so that the somatic voltage fluctuates with an SD of about **6 mV**. The noise is fast compared to the 16 ms membrane, so the membrane smooths most of it out, and the current SD needs to be much larger than you might expect. Here we sweep $\sigma$ and read off the value that gives 6 mV.

In [ ]:
sigmas = np.arange(0, 701, 100)
quiet = np.zeros(int(2000 / 0.1))  # 2 s, no mean input, so the neuron stays below threshold
sd = [simulate_soma(quiet, sigma_noise_pA=s, seed=1)["u"][2000:].std() for s in sigmas]

slope = np.polyfit(sigmas, sd, 1)[0]
sigma_6mV = 6 / slope
print(f"σ ≈ {sigma_6mV:.0f} pA gives a membrane SD of 6 mV")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.plot(sigmas, sd, "o-", color=COLORS["soma"], ms=5)
ax.axhline(6, color=INK_SOFT, lw=0.8, ls="--")
ax.text(sigmas[0], 6.2, "paper's target: 6 mV", fontsize=8, color=INK_SOFT)
ax.set_xlabel("noise σ (pA)")
ax.set_ylabel("SD of u_s (mV)")
ax.set_title("Noise size vs voltage fluctuations")
plt.tight_layout()
plt.show()

In [ ]:
# With noise, the same sub-rheobase 400 pA step now fires irregularly.
r = simulate_soma(step(400, 50 * ms, 950 * ms, 1000 * ms), sigma_noise_pA=sigma_6mV, seed=3)
plot_run(r, f"400 pA step + noise (σ = {sigma_6mV:.0f} pA)")

## 6 · Play

Move the sliders and press **Run Interact**. Each run takes about a second. Things to try:

- Flip `b_w_s` to **+200 pA**. The brake becomes an accelerator and firing runs away.
- Make the plateau shorter (5 ms) or weaker (500 pA). When does it stop producing a burst?
- Set `tau_w_s` to 20 ms. Adaptation wears off fast, so intervals stop growing.

In [ ]:
from ipywidgets import interact_manual, FloatSlider, IntSlider

@interact_manual(
    background_pA=IntSlider(500, 0, 1500, 50),
    plateau_pA=IntSlider(1300, 0, 2000, 100),
    plateau_ms=IntSlider(25, 0, 60, 5),
    b_w_s_pA=IntSlider(-200, -400, 200, 25),
    tau_w_s_ms=IntSlider(100, 10, 300, 10),
    noise_pA=IntSlider(0, 0, 700, 50),
)
def play(background_pA, plateau_pA, plateau_ms, b_w_s_pA, tau_w_s_ms, noise_pA):
    duration = 400 * ms
    I = step(background_pA, 0 * ms, 400 * ms, duration) + plateau(plateau_pA, 200 * ms, plateau_ms * ms, duration)
    r = simulate_soma(I, params={"b_w_s": b_w_s_pA * pA, "tau_w_s": tau_w_s_ms * ms}, sigma_noise_pA=noise_pA)
    plot_run(r, "Your settings")

## What to take into notebook 02

- The soma by itself **cannot burst**: steady input gives regular or slowing spikes, never a tight cluster.
- A burst needs a **strong, brief extra current**: about 1.3 nA for 10–40 ms. Notebook 02 builds the dendrite that produces it.